In [ ]:
import pandas as pd
import numpy as np
import plotly as pt

import matplotlib.pyplot as plt

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import make_scorer, roc_auc_score, average_precision_score, precision_score
from sklearn.preprocessing import OrdinalEncoder

#For OS
import os

In [ ]:
Test_Data = pd.read_csv('data/raw/application_test.csv')
Train_Data = pd.read_csv('data/raw/application_train.csv')

In [ ]:
print(Test_Data.head())


In [ ]:
print(Train_Data.head())

In [ ]:
print("Train_Data:", Train_Data.shape)
print("Test_Data:", Test_Data.shape)

In [ ]:
print(set(Train_Data.columns) - set(Test_Data.columns))
print(set(Test_Data.columns) - set(Train_Data.columns))

In [ ]:
#Train data info
Train_Data.info()

In [ ]:
Train_Data.sample(5)

In [ ]:
Train_Data.tail()

In [ ]:
#Column understand
print(Train_Data.columns.tolist())

In [ ]:
#Column in list
for i, col in enumerate(Train_Data.columns, start=1):
    print(i, col)

In [ ]:
# numerical and categorical
numeric_cols = Train_Data.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = Train_Data.select_dtypes(
    include=["object"]
).columns

print("Numerical columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

In [ ]:
print("Duplicate rows:", Train_Data.duplicated().sum())
print("Duplicate SK_ID_CURR:", Train_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
#missing values
missing = Train_Data.isna().sum().sort_values(ascending=False)

print(missing.head(20))

In [ ]:
#in percentage
missing_pct = (
    Train_Data.isna().mean() * 100
).sort_values(ascending=False)

print(missing_pct.head(20))

In [ ]:
#Target distribution
print(
    Train_Data["TARGET"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:

Train_Data["TARGET"].value_counts().plot(kind="bar")

plt.title("Target Distribution")
plt.xlabel("TARGET")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
Train_Data.describe().T

In [ ]:
#Relationship with Target
Train_Data.groupby("TARGET")["AMT_INCOME_TOTAL"].median()

In [ ]:
#Distribution
sns.boxplot(
    data=Train_Data,
    x="TARGET",
    y="AMT_INCOME_TOTAL"
)

plt.show()

In [ ]:
Train_Data.isnull().sum()

In [ ]:
missing = Train_Data.isnull().sum().sort_values(ascending=False)

for column, count in missing.items():
    print(f"{column}: {count}")

In [ ]:
missing = Train_Data.isnull().sum().sort_values(ascending=False)

for column, count in missing.items():
    if count > 0:
        print(f"{column}: {count}")

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": Train_Data.isnull().sum(),
    "Missing_Percentage": Train_Data.isnull().mean() * 100
})

missing = missing[missing["Missing_Count"] > 0] \
                 .sort_values("Missing_Percentage", ascending=False)

print(missing)

In [ ]:
print("Duplicate rows:", Train_Data.duplicated().sum())
print("Duplicate SK_ID_CURR:", Train_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
print("Test duplicate rows:", Test_Data.duplicated().sum())
print("Test duplicate SK_ID_CURR:", Test_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
print(Train_Data.dtypes.value_counts())

In [ ]:
numeric_cols = Train_Data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = Train_Data.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

In [ ]:
Train_Data["AMT_INCOME_TOTAL"].describe()

In [ ]:
Train_Data["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
Train_Data["OCCUPATION_TYPE"].value_counts(dropna=False)

In [ ]:
pd.crosstab(
    Train_Data["NAME_CONTRACT_TYPE"],
    Train_Data["TARGET"],
    normalize="index"
) * 100

#checking for summary
#numeric variable eda

In [ ]:
Train_Data["AMT_INCOME_TOTAL"].describe()

In [ ]:
Train_Data["AMT_CREDIT"].describe()

In [ ]:
Train_Data["AMT_ANNUITY"].describe()

In [ ]:
Train_Data["AMT_GOODS_PRICE"].describe()

In [ ]:
Train_Data["DAYS_BIRTH"].describe()

In [ ]:
(Train_Data['DAYS_BIRTH']/-365).describe()

In [ ]:
## Distribution of client age 
fig, ax = plt.subplots(figsize=(12, 7))

sns.histplot(
    Train_Data["DAYS_BIRTH"] / -365,
    bins=30,
    kde=False,
    ax=ax
)

ax.set_xlabel("Age of the client (Years)")
ax.set_ylabel("Number of clients")
ax.set_title("Distribution of Client Age")

plt.show()

Interpretation: People in the age range 30-40 years are the most applicants. Which seems pretty normal.

In [ ]:
Train_Data["DAYS_EMPLOYED"].describe()

In [ ]:
(Train_Data['DAYS_EMPLOYED']/365).describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12,7))
sns.histplot(Train_Data['DAYS_EMPLOYED']/365,kde=False)
plt.xlabel("Time before the loan application the persons started current employment(in years)")

In [ ]:
Train_Data["CNT_CHILDREN"].describe()

In [ ]:
Train_Data["CNT_FAM_MEMBERS"].describe()

#Categorical variables eda

In [ ]:
categorical_cols = Train_Data.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    print(col)


In [ ]:

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(Train_Data[col].value_counts(dropna=False))

In [ ]:
Train_Data["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
corr = Train_Data[numeric_cols].corr()

In [ ]:
target_corr = (
    corr["TARGET"]
    .drop("TARGET")
    .sort_values()
)

print(target_corr)

In [ ]:
for i, col in enumerate(Train_Data.columns, start=1):
    print(i, col)

##Preprocessing


In [ ]:
#Keep ID separately
Train_ID = Train_Data["SK_ID_CURR"]

In [ ]:
#Target
y = Train_Data["TARGET"]

In [ ]:
#Features
X = Train_Data.drop(columns=["SK_ID_CURR", "TARGET"])

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
#Split into training and validation
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

In [ ]:
#identify numerical and categorical features
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Total features:", len(num_cols) + len(cat_cols))

In [ ]:
cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist() + ["CNT_CHILDREN","CNT_FAM_MEMBERS", "REGION_RATING_CLIENT","REGION_RATING_CLIENT_W_CITY"]

print(cat_cols)

ord_cols = ["AMT_REQ_CREDIT_BUREAU_HOUR","AMT_REQ_CREDIT_BUREAU_DAY","AMT_REQ_CREDIT_BUREAU_WEEK","AMT_REQ_CREDIT_BUREAU_MON","AMT_REQ_CREDIT_BUREAU_QRT","AMT_REQ_CREDIT_BUREAU_YEAR"]

binary_cols = ["FLAG_MOBIL","FLAG_EMP_PHONE","FLAG_WORK_PHONE","FLAG_CONT_MOBILE","FLAG_PHONE","FLAG_EMAIL","REG_REGION_NOT_LIVE_REGION","REG_REGION_NOT_WORK_REGION","LIVE_REGION_NOT_WORK_REGION",
"REG_CITY_NOT_LIVE_CITY",
"REG_CITY_NOT_WORK_CITY",
"LIVE_CITY_NOT_WORK_CITY","FLAG_DOCUMENT_2","FLAG_DOCUMENT_3",
"FLAG_DOCUMENT_4","FLAG_DOCUMENT_5","FLAG_DOCUMENT_6",
"FLAG_DOCUMENT_7","FLAG_DOCUMENT_8","FLAG_DOCUMENT_9","FLAG_DOCUMENT_10","FLAG_DOCUMENT_11","FLAG_DOCUMENT_12","FLAG_DOCUMENT_13",
"FLAG_DOCUMENT_14","FLAG_DOCUMENT_15","FLAG_DOCUMENT_16","FLAG_DOCUMENT_17",
"FLAG_DOCUMENT_18","FLAG_DOCUMENT_19","FLAG_DOCUMENT_20","FLAG_DOCUMENT_21"]

numeric_cols =X_train.drop(columns=cat_cols+ord_cols+binary_cols).columns.tolist()
print(numeric_cols)

print(len(cat_cols)+len(numeric_cols)+len(ord_cols)+len(binary_cols))

In [ ]:
#proportion of defaults is similar in train and validation.
print("Training target:")
print(y_train.value_counts(normalize=True))

print("\nValidation target:")
print(y_val.value_counts(normalize=True))

## Preprocessing pipeline

In [ ]:
#Create preprocessing pipelines
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

#Categorical features
#Fill missing values with most frequent category
#One-hot encode
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ))
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="ignore"
    ))
])


#Combine them
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num_cols", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, cat_cols),
    ("binary", binary_pipeline, binary_cols),
    ("ordinal", ordinal_pipeline, ord_cols)
])

In [ ]:
print("Shape:", Train_Data.shape)

In [ ]:
print("SK_ID_CURR exists:", "SK_ID_CURR" in Train_Data.columns)
print("TARGET exists:", "TARGET" in Train_Data.columns)

In [ ]:
print("Duplicate IDs:", Train_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
print("Duplicate rows:", Train_Data.duplicated().sum())

In [ ]:
print(Train_Data["TARGET"].value_counts())

In [ ]:
print(Train_Data["TARGET"].unique())

In [ ]:
print(Train_Data.dtypes.value_counts())

In [ ]:
print(Train_Data.info())

In [ ]:
missing = Train_Data.isna().sum()

print("Total missing values:", missing.sum())
print("Columns with missing values:", (missing > 0).sum())

In [ ]:
print("Train_Data:", Train_Data.shape)
print("X:", X.shape)
print("y:", y.shape)
print("Train_ID:", Train_ID.shape)

In [ ]:
print("Original columns:", Train_Data.shape[1])
print("ID + TARGET + features:", 1 + 1 + X.shape[1])

In [ ]:
print(len(num_cols))
print(len(cat_cols))
print(len(num_cols) + len(cat_cols))

### Modeling

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train, y_train)

In [ ]:
y_pred_dummy = dummy_model.predict(X_val)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_val, y_pred_dummy))
print("Precision:", precision_score(y_val, y_pred_dummy, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_dummy, zero_division=0))
print("F1:", f1_score(y_val, y_pred_dummy, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_dummy))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_dummy, zero_division=0))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
         random_state=42
    ))
])

In [ ]:
baseline_model.fit(X_train, y_train)

In [ ]:
y_val_pred = baseline_model.predict(X_val)

In [ ]:
y_val_prob = baseline_model.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("ROC-AUC:",
      roc_auc_score(y_val, y_val_prob))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    baseline_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
from sklearn.tree import DecisionTreeClassifier


dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
dt_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = dt_pipeline.predict(X_val)
y_prob = dt_pipeline.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print("Recall:", recall_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("F1:", f1_score(y_val, y_pred))
print("F2:", fbeta_score(y_val, y_pred, beta=2))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))

In [ ]:
param_grid_small = {
    "model__max_depth": [5, 10, 15],
    "model__min_samples_leaf": [5, 10],
    "model__class_weight": [None, "balanced"]
}

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

dt_random = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_grid_small,
    n_iter=3,
    scoring="roc_auc",
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

dt_random.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(dt_random.best_params_)

print("Best CV ROC-AUC:")
print(dt_random.best_score_)

In [ ]:
best_dt = dt_random.best_estimator_

y_pred_dt_tuned = best_dt.predict(X_val)
y_prob_dt_tuned = best_dt.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("ROC-AUC:", roc_auc_score(y_val, y_prob_dt_tuned))
print("Recall:", recall_score(y_val, y_pred_dt_tuned))
print("Precision:", precision_score(y_val, y_pred_dt_tuned))
print("F1:", f1_score(y_val, y_pred_dt_tuned))
print("F2:", fbeta_score(y_val, y_pred_dt_tuned, beta=2))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_dt_tuned))

In [ ]:
print(dt_random.best_params_)

In [ ]:
dt_pipeline_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
param_grid_small = {
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_leaf": [5, 10, 20]
}

In [ ]:
dt_random_balanced = RandomizedSearchCV(
    estimator=dt_pipeline_balanced,
    param_distributions=param_grid_small,
    n_iter=6,
    scoring="roc_auc",
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

dt_random_balanced.fit(X_train, y_train)

In [ ]:
print(dt_random_balanced.best_params_)
print(dt_random_balanced.best_score_)

In [ ]:
best_dt_balanced = dt_random_balanced.best_estimator_

y_pred_balanced = best_dt_balanced.predict(X_val)
y_prob_balanced = best_dt_balanced.predict_proba(X_val)[:, 1]

In [ ]:
print("ROC-AUC:", roc_auc_score(y_val, y_prob_balanced))
print("Recall:", recall_score(y_val, y_pred_balanced))
print("Precision:", precision_score(y_val, y_pred_balanced))
print("F1:", f1_score(y_val, y_pred_balanced))
print("F2:", fbeta_score(y_val, y_pred_balanced, beta=2))
print(confusion_matrix(y_val, y_pred_balanced))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, 15],
    "model__min_samples_leaf": [5, 10, 20],
    "model__max_features": ["sqrt", "log2"]
}

In [ ]:

cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=5,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
rf_random.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(rf_random.best_params_)

print("Best CV ROC-AUC:")
print(rf_random.best_score_)

In [ ]:
best_rf = rf_random.best_estimator_

y_pred_rf = best_rf.predict(X_val)
y_prob_rf = best_rf.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("ROC-AUC:", roc_auc_score(y_val, y_prob_rf))
print("Recall:", recall_score(y_val, y_pred_rf))
print("Precision:", precision_score(y_val, y_pred_rf))
print("F1:", f1_score(y_val, y_pred_rf))
print("F2:", fbeta_score(y_val, y_pred_rf, beta=2))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Recall: {recall_score(y_val, y_pred_threshold):.3f} | "
        f"Precision: {precision_score(y_val, y_pred_threshold):.3f} | "
        f"F1: {f1_score(y_val, y_pred_threshold):.3f} | "
        f"F2: {fbeta_score(y_val, y_pred_threshold, beta=2):.3f}"
    )

In [ ]:
best_rf = rf_random.best_estimator_

rf_model = best_rf.named_steps["model"]
preprocessor_fitted = best_rf.named_steps["preprocessor"]

feature_names = preprocessor_fitted.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(20)

In [ ]:
import matplotlib.pyplot as plt

top_15 = importance.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_15["Feature"], top_15["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 15 Features – Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.arange(0.40, 0.501, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Recall": recall_score(y_val, y_pred_threshold),
        "Precision": precision_score(y_val, y_pred_threshold),
        "F1": f1_score(y_val, y_pred_threshold),
        "F2": fbeta_score(y_val, y_pred_threshold, beta=2)
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results.sort_values(
    "F2",
    ascending=False
).head(10)

In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_pipeline.predict(X_val)

y_prob_xgb = xgb_pipeline.predict_proba(X_val)[:, 1]

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("ROC-AUC:", roc_auc_score(y_val, y_prob_xgb))
print("Recall:", recall_score(y_val, y_pred_xgb))
print("Precision:", precision_score(y_val, y_pred_xgb))
print("F1:", f1_score(y_val, y_pred_xgb))
print("F2:", fbeta_score(y_val, y_pred_xgb, beta=2))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb))

In [ ]:
thresholds = np.arange(0.20, 0.61, 0.05)

for threshold in thresholds:
    y_pred_threshold = (y_prob_xgb >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Recall: {recall_score(y_val, y_pred_threshold):.3f} | "
        f"Precision: {precision_score(y_val, y_pred_threshold):.3f} | "
        f"F1: {f1_score(y_val, y_pred_threshold):.3f} | "
        f"F2: {fbeta_score(y_val, y_pred_threshold, beta=2):.3f}"
    )

In [ ]:
##Test Data

In [ ]:
print(Test_Data.shape)
print(Test_Data.head())
print(Test_Data.columns.tolist())

In [ ]:
print("TARGET" in Test_Data.columns)

In [ ]:
xgb_pipeline

In [ ]:
test_prob = xgb_pipeline.predict_proba(Test_Data)[:, 1]

In [ ]:
print(test_prob[:10])
print(test_prob.min())
print(test_prob.max())

In [ ]:
test_pred = (test_prob >= 0.50).astype(int)

In [ ]:
submission = pd.DataFrame({
    "SK_ID_CURR": Test_Data["SK_ID_CURR"],
    "TARGET": test_prob
})

In [ ]:
print(submission.head())
print(submission.shape)

In [ ]:
Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(...))
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)

In [ ]:
xgb_pipeline.predict(X_val)